#Clone the Datasets from Github

In [1]:
!git clone https://github.com/DaoPhang/Algo-Project.git
!pip install openpyxl tabulate --quiet

Cloning into 'Algo-Project'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 151 (delta 66), reused 68 (delta 25), pack-reused 12 (from 1)
Receiving objects: 100% (151/151), 53.47 MiB | 37.43 MiB/s, done.
Resolving deltas: 100% (72/72), done.


# Part 7 — The Phantom Dice

## Step 1 — Mission Problem and Candidate Algorithms

**Resource:** Infiltration sector — one sector the team will physically enter.

**Constraint:** Cypher Nexus has flagged certain sectors as monitored. If the team repeatedly uses a predictable entry point, the enemy can learn the pattern and intercept them.

**Goal:** Select a real movement sector that is safe, deception-aware, and non-deterministic, while also selecting a decoy sector to misdirect surveillance.

### Chosen Algorithm

| Status | Algorithm | Main Idea |
|---|---|---|
| CHOSEN | Dual-Objective Sector Scoring with Weighted Prefix Draw | Combines danger score, safety weight, deception bonus, prediction penalty, and prefix-sum random draw. |

## Step 2 — Dataset Loading and Preview

In [2]:
# ---------------------------------------------------------------------
# 0. Install & Imports
# ---------------------------------------------------------------------

import os
import random
import time
import openpyxl
import pandas as pd
from tabulate import tabulate


In [3]:
# ─────────────────────────────────────────────────────────────────────
# 1. Dataset Loading and Preview
# ─────────────────────────────────────────────────────────────────────
EXCEL_PATH = None
search_roots = ["/content/Algo-Project", os.getcwd()]
for search_root in search_roots:
    if not os.path.exists(search_root):
        continue
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if "7" in file and file.endswith(".xlsx") and not file.startswith("."):
                EXCEL_PATH = os.path.join(root, file)
                break
        if EXCEL_PATH is not None:
            break
    if EXCEL_PATH is not None:
        break

if EXCEL_PATH is None:
    raise FileNotFoundError("Part 7 dataset not found. Check your repo structure.")
print(f"Dataset found: {EXCEL_PATH}")

SHEET_NAME = "B"
wb = openpyxl.load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

sector_columns = [
    "Sector",
    "Patrol_Frequency",
    "Thermal_Scan_Level",
    "Drone_Coverage",
    "Predicted_By_Enemy",
    "Decoy_Value",
]

sectors = []
for row in ws.iter_rows(min_row=3, values_only=True):
    if row[1] is None or row[1] == "Sector":
        continue
    sectors.append({
        "Sector"            : row[1],
        "Patrol_Frequency"  : row[2],
        "Thermal_Scan_Level": row[3],
        "Drone_Coverage"    : row[4],
        "Predicted_By_Enemy": row[5],
        "Decoy_Value"       : row[6],
    })

sector_df = pd.DataFrame(sectors, columns=sector_columns)
print(f"Loaded {len(sector_df)} sector records from Sheet '{SHEET_NAME}'\n")
print("=" * 95)
print("  PART 7 — THE PHANTOM DICE: DATASET PREVIEW")
print("=" * 95)
print(tabulate(sector_df[sector_columns], headers="keys", tablefmt="rounded_outline", showindex=False))

Dataset found: /content/Algo-Project/Datasets/Part 7.xlsx
Loaded 8 sector records from Sheet 'B'

  PART 7 — THE PHANTOM DICE: DATASET PREVIEW
╭──────────┬────────────────────┬──────────────────────┬──────────────────┬──────────────────────┬───────────────╮
│ Sector   │   Patrol_Frequency │   Thermal_Scan_Level │   Drone_Coverage │ Predicted_By_Enemy   │   Decoy_Value │
├──────────┼────────────────────┼──────────────────────┼──────────────────┼──────────────────────┼───────────────┤
│ S1       │                  2 │                    2 │                2 │ Yes                  │             4 │
│ S2       │                  5 │                    4 │                5 │ No                   │             8 │
│ S3       │                  7 │                    6 │                8 │ Yes                  │             3 │
│ S4       │                  4 │                    3 │                4 │ No                   │             7 │
│ S5       │                  6 │                   

## Step 3 — Define Helper Functions

In [4]:
def _as_sector_frame(sector_data):
    if isinstance(sector_data, pd.DataFrame):
        frame = sector_data.copy()
    else:
        frame = pd.DataFrame(sector_data)
    required = [
        "Sector", "Patrol_Frequency", "Thermal_Scan_Level",
        "Drone_Coverage", "Predicted_By_Enemy", "Decoy_Value"
    ]
    missing = [col for col in required if col not in frame.columns]
    if missing:
        raise ValueError(f"Missing required Part 7 columns: {missing}")
    for col in ["Patrol_Frequency", "Thermal_Scan_Level", "Drone_Coverage", "Decoy_Value"]:
        frame[col] = pd.to_numeric(frame[col])
    return frame[required].reset_index(drop=True)

def _draw_from_prefix(prefix_values):
    r = random.random()
    selected_idx = len(prefix_values) - 1
    for idx, threshold in enumerate(prefix_values):
        if r <= threshold:
            selected_idx = idx
            break
    return r, selected_idx

## Step 4 — Algorithm 1: Dual-Objective Sector Scoring with Weighted Prefix Draw (Chosen)

In [5]:
def dual_objective_prefix_draw(sector_data):
    t0 = time.perf_counter()
    table = _as_sector_frame(sector_data)

    table["Danger_Score"] = (
        table["Patrol_Frequency"]
        + table["Thermal_Scan_Level"]
        + table["Drone_Coverage"]
    )
    max_decoy = table["Decoy_Value"].max()
    table["Safety_Weight"] = 1 / table["Danger_Score"]
    table["Deception_Bonus"] = 1 + (table["Decoy_Value" ] / max_decoy if max_decoy else 0)
    table["Composite_Score"] = table["Safety_Weight"] * table["Deception_Bonus"]
    table.loc[table["Predicted_By_Enemy"] == "Yes", "Composite_Score"] /= 2
    table["Selection_Probability"] = table["Composite_Score"] / table["Composite_Score"].sum()
    table["Prefix_Probability"] = table["Selection_Probability"].cumsum()

    r, selected_idx = _draw_from_prefix(table["Prefix_Probability"])
    selected_real_sector = table.iloc[selected_idx].to_dict()
    real_sector_name = selected_real_sector["Sector"]

    predicted_candidates = table[
        (table["Predicted_By_Enemy"] == "Yes")
        & (table["Sector"] != real_sector_name)
    ]
    if predicted_candidates.empty:
        predicted_candidates = table[table["Sector"] != real_sector_name]
    decoy_sector = predicted_candidates.sort_values(
        ["Decoy_Value", "Composite_Score"], ascending=[False, False]
    ).iloc[0].to_dict()

    elapsed = time.perf_counter() - t0
    return {
        "selected_real_sector": selected_real_sector,
        "decoy_sector": decoy_sector,
        "probability_table": table,
        "random_value": r,
        "elapsed": elapsed,
    }

## Step 5 — Run the Chosen Algorithm

In [6]:
# ─────────────────────────────────────────────────────────────────────
# 3. Run the Chosen Algorithm
# ─────────────────────────────────────────────────────────────────────
print("Running the chosen Part 7 algorithm...\n")
dual_result = dual_objective_prefix_draw(sector_df)

real_sector = dual_result["selected_real_sector"]
decoy_sector = dual_result["decoy_sector"]

Running the chosen Part 7 algorithm...



## Step 6 — Chosen Algorithm Output and Probability Table

In [7]:
# ─────────────────────────────────────────────────────────────────────
# 4. Chosen Algorithm Output: Dual-Objective Sector Scoring
# ─────────────────────────────────────────────────────────────────────
chosen_table = dual_result["probability_table"]
chosen_display_columns = [
    "Sector",
    "Patrol_Frequency",
    "Thermal_Scan_Level",
    "Drone_Coverage",
    "Danger_Score",
    "Predicted_By_Enemy",
    "Decoy_Value",
    "Safety_Weight",
    "Deception_Bonus",
    "Composite_Score",
    "Selection_Probability",
    "Prefix_Probability",
]

print("=" * 120)
print("  CHOSEN ALGORITHM: DUAL-OBJECTIVE SECTOR SCORING WITH WEIGHTED PREFIX DRAW")
print("=" * 120)
print("danger_score = Patrol_Frequency + Thermal_Scan_Level + Drone_Coverage")
print("safety_weight = 1 / danger_score")
print("deception_bonus = 1 + (Decoy_Value / max_Decoy_Value)")
print("score = safety_weight × deception_bonus")
print("if Predicted_By_Enemy == Yes: score = score / 2")
print("probability = score / total_score\n")
print(tabulate(
    chosen_table[chosen_display_columns],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False,
    floatfmt=".4f",
))
print("\nNote: Prefix_Probability of last sector = 1.0000 confirms all probabilities sum correctly to 1.")

  CHOSEN ALGORITHM: DUAL-OBJECTIVE SECTOR SCORING WITH WEIGHTED PREFIX DRAW
danger_score = Patrol_Frequency + Thermal_Scan_Level + Drone_Coverage
safety_weight = 1 / danger_score
deception_bonus = 1 + (Decoy_Value / max_Decoy_Value)
score = safety_weight × deception_bonus
if Predicted_By_Enemy == Yes: score = score / 2
probability = score / total_score

╭──────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────┬──────────────────────┬───────────────┬─────────────────┬───────────────────┬───────────────────┬─────────────────────────┬──────────────────────╮
│ Sector   │   Patrol_Frequency │   Thermal_Scan_Level │   Drone_Coverage │   Danger_Score │ Predicted_By_Enemy   │   Decoy_Value │   Safety_Weight │   Deception_Bonus │   Composite_Score │   Selection_Probability │   Prefix_Probability │
├──────────┼────────────────────┼──────────────────────┼──────────────────┼────────────────┼──────────────────────┼───────────────┼─────────────────┼─────────────────

In [8]:
# ---------------------------------------------------------------------
# 5. Chosen Algorithm Result: Real Sector and Decoy Sector
# ---------------------------------------------------------------------
print("=" * 95)
print("  SELECTED REAL SECTOR & DECOY SECTOR")
print("=" * 95)
print(f"Random value r                 : {dual_result['random_value']:.6f}")
print(f"Selected real sector           : {real_sector['Sector']}")
print(f"Real sector danger score       : {real_sector['Danger_Score']}")
print(f"Real sector Decoy_Value        : {real_sector['Decoy_Value']}")
print(f"Real sector predicted status   : {real_sector['Predicted_By_Enemy']}")
print(f"Decoy sector                   : {decoy_sector['Sector']}")
print(f"Decoy sector danger score      : {decoy_sector['Danger_Score']}")
print(f"Decoy sector Decoy_Value       : {decoy_sector['Decoy_Value']}")
print(f"Decoy sector predicted status  : {decoy_sector['Predicted_By_Enemy']}")
print(f"Time taken                     : {dual_result['elapsed'] * 1000:.4f} ms")
print("Note: Because this algorithm is non-deterministic, the selected real sector may differ each run.")


  SELECTED REAL SECTOR & DECOY SECTOR
Random value r                 : 0.827913
Selected real sector           : S7
Real sector danger score       : 22
Real sector Decoy_Value        : 2
Real sector predicted status   : Yes
Decoy sector                   : S1
Decoy sector danger score      : 6
Decoy sector Decoy_Value       : 4
Decoy sector predicted status  : Yes
Time taken                     : 20.2896 ms
Note: Because this algorithm is non-deterministic, the selected real sector may differ each run.


## Step 7 — Time and Space Complexity Analysis

**Dual-Objective Sector Scoring with Weighted Prefix Draw:**

`T_total = O(n) + O(n) + O(n) + O(n) + O(n) + O(n) = O(n)`

`Space = O(n)`